# Cvičení 4: Rozpoznávání ručně psaných číslic

- Cílem cvičení je natrénovat klasifikátor ručně psaných číslic na datasetu [digits](https://scikit-learn.org/stable/datasets/toy_dataset.html#digits-dataset) a vyhodnotit ho na validačních datech.
- Vstupem je šedotónový obrázek 8×8 pixelů a úkolem říci, která z 10 číslic 0 až 9 (tříd) je na něm.
- Použitý je **lineární model**, **softmax křížová entropie** a **metoda největšího spádu**, vše implementované ručně pouze pomocí numpy.

1. Dataset
2. Rozdělení dat
3. Predikce a přesnost
4. Loss a gradient
5. Trénování
6. Vyhodnocení
7. Bonus: vlastní obrázek

Obecné pokyny:
- Dataset načtěte funkcí `load_digits` z knihovny [scikit-learn](https://scikit-learn.org/). Jinak knihovny strojového učení (scikit-learn, PyTorch apod.) **nejsou povoleny**, vše implementujte v [numpy](https://numpy.org/).
- V bonusu můžete pro zpracování obrazu používat [OpenCV](https://docs.opencv.org/) nebo např. [scikit-image](https://scikit-image.org/).
- Výsledky vykreslujte pomocí [matplotlib](https://matplotlib.org/).
- Svůj kód pište na místa označená komentářem `# TODO: implement`.
- Většina sekcí obsahuje vzorový výstup. Váš výsledek nemusí být na pixel stejný, ale měl by vypadat podobně.
- Za většinou bodů je buňka s testy ze souboru `tests/test_digit_recognition.py`.

In [ ]:
import sys

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from sklearn.datasets import load_digits

if ".." not in sys.path:
    sys.path.append("..")
from tests import test_digit_recognition

## 1. Dataset

- Dataset obsahuje 1797 číslic od 43 pisatelů. Každá číslice byla naskenována jako binární obrázek 32×32, zmenšena tak, aby vyplnila obrázek, a převedena na obrázek 8×8, kde každý pixel je počet černých pixelů v bloku 4×4 (hodnoty 0 až 16).
- `load_digits()` vrátí objekt s atributy `images` (tvar `(1797, 8, 8)`), `data` (tytéž obrázky jako vektory, tvar `(1797, 64)`) a `target` (třídy 0 až 9).
- Vykreslete několik vzorků každé třídy.
- Připravte vstupy klasifikátoru `inputs` (tvar `(1797, 64)`) a třídy `labels`. Doporučené je vzorky převést na `float` a přeškálovat do rozsahu 0 až 1.

Vzorový výstup:

<figure class="image">
  <img src="../figures/digit_recognition-expected_outputs_dataset.png" alt="Vzorový výstup" style="width: 60%"/>
</figure>

In [ ]:
digits = load_digits()
print(digits.images.shape, digits.data.shape, digits.target.shape)

In [ ]:
########################################
# TODO: implement

# Vykresleni
# ...

########################################

In [ ]:
########################################
# TODO: implement

# inputs, labels = ...

########################################

## 2. Rozdělení dat

- Implementujte funkci `split_data(inputs, labels, val_fraction, seed)`, která náhodně rozdělí vzorky na trénovací a validační v daném poměru a vrátí `(inputs_train, inputs_val, labels_train, labels_val)`.
- Funkcí rozdělte `inputs` a `labels` z bodu 1 (20 % validačních dat).
- `TestSplit` vyzkouší funkci na umělých datech, `TestData` zkontroluje, že `inputs_train`, `inputs_val` jsou (případně přeškálované) vzorky datasetu se správnými třídami a každý vzorek je použit právě jednou.

In [ ]:
def split_data(
    inputs: NDArray,
    labels: NDArray[np.int64],
    val_fraction: float = 0.2,
    seed: int = 0,
) -> tuple[NDArray, NDArray, NDArray[np.int64], NDArray[np.int64]]:
    """
    Randomly split the samples into training and validation sets.

    Args:
        inputs: Samples of shape (N, D).
        labels: Classes of shape (N,).
        val_fraction: Fraction of the samples in the validation set.
        seed: Seed of the random number generator.
    Returns:
        inputs_train, inputs_val, labels_train, labels_val.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
########################################
# TODO: implement

# inputs_train, inputs_val, labels_train, labels_val = ...

########################################

In [ ]:
test_digit_recognition.TestSplit.eval(split_fn=split_data)
test_digit_recognition.TestData.eval(
    inputs_train=inputs_train, inputs_val=inputs_val, labels_train=labels_train, labels_val=labels_val
)

## 3. Predikce a přesnost

- Lineární klasifikátor spočítá pro každý vzorek skóre tříd jako lineární (afinní) funkci
  $$
  \boldsymbol{s}_n = \boldsymbol{x}_n \cdot \boldsymbol{w} + \boldsymbol{b}
  $$
  kde
  - $\boldsymbol{x}_n = [x_{n,1}, \ldots, x_{n,D}]$ je $n$-tý vzorek (řádek `inputs`) reprezentovaný jako řádkový vektor s rozměrem $D = 64$,
  - $\boldsymbol{w} = [w_{d,k}]$ je matice vah (`weights`) s rozměrem $D \times K$,
  - $\boldsymbol{b} = [b_1, \ldots, b_K]$ je vektor biasů (`bias`) s rozměrem $K$,
  - $\boldsymbol{s}_n = [s_{n,1}, \ldots, s_{n,K}]$ je řádkový vektor skóre $n$-tého vzorku pro každou z $K = 10$ tříd.
- Predikovaná třída je ta s nejvyšším skóre
  $$
  \hat{y}_n = \arg\max_{k} s_{n,k}
  $$
- Přesnost (accuracy) $a$ je podíl správně klasifikovaných vzorků
  $$
  a = \frac{1}{N} \cdot \sum_{n=1}^{N}{\mathbb{1}(\hat{y}_n = y_n)}
  $$
  kde
  - $y_n$ je správná třída $n$-tého vzorku,
  - $\mathbb{1}(\cdot) = 1$, pokud je podmínka $\cdot$ splněna, jinak $\mathbb{1}(\cdot) = 0$.
- Implementujte funkci `predict(inputs, weights, bias)`, která vrátí predikované třídy všech řádků matice `inputs`.
- Implementujte funkci `accuracy(predicted, labels)`, která vrátí přesnost (číslo 0 až 1).
- Obě funkce implementujte bez cyklů, pouze maticovými operacemi numpy.

In [ ]:
def predict(inputs: NDArray[np.float64], weights: NDArray[np.float64], bias: NDArray[np.float64]) -> NDArray[np.int64]:
    """
    Predict the classes of the samples by the linear classifier.

    Args:
        inputs: Samples of shape (N, D).
        weights: Weights of shape (D, K).
        bias: Bias of shape (K,).
    Returns:
        Predicted classes of shape (N,).
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################


def accuracy(predicted: NDArray[np.int64], labels: NDArray[np.int64]) -> float:
    """
    Fraction of correctly classified samples.

    Args:
        predicted: Predicted classes of shape (N,).
        labels: True classes of shape (N,).
    Returns:
        Accuracy in the range 0 to 1.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
test_digit_recognition.TestPredict.eval(predict_fn=predict, accuracy_fn=accuracy)

## 4. Loss a gradient

- Softmax klasifikátor převede skóre na pravděpodobnosti tříd
  $$
  p_{n,c} = \frac{\exp{s_{n,c}}}{\sum_{k=1}^{K}{\exp{s_{n,k}}}}
  $$
  a optimalizuje křížovou entropii, která se pro celá data počítá jako *průměr* přes vzorky
  $$
  L = -\frac{1}{N} \cdot \sum_{n=1}^{N}{\log p_{n,y_n}}
  $$
- Implementujte funkci `softmax_loss(inputs, labels, weights, bias)`, která vrátí trojici `(loss, grad_weights, grad_bias)`:
  - `loss` je hodnota lossu $L$,
  - `grad_weights` a `grad_bias` jsou gradienty $\frac{\partial L}{\partial \boldsymbol{w}}$ a $\frac{\partial L}{\partial \boldsymbol{b}}$ se stejnými rozměry jako `weights` a `bias`.
- Nápověda: gradient lossu vůči skóre je
  $$
  \frac{\partial L}{\partial s_{n,c}} = \frac{1}{N} \cdot \left(p_{n,c} - \mathbb{1}(c = y_n)\right)
  $$
  Gradienty vůči $\boldsymbol{w}$ a $\boldsymbol{b}$ z něj odvoďte pomocí řetízkového pravidla.
- Funkci implementujte bez cyklů. Pozor na přetečení `np.exp` pro velká skóre (proč nevadí odečíst od skóre jeho maximum?).
- Test porovná loss s referenční hodnotou a gradienty s numerickým gradientem vašeho lossu (centrální diference).

In [ ]:
def softmax_loss(
    inputs: NDArray[np.float64],
    labels: NDArray[np.int64],
    weights: NDArray[np.float64],
    bias: NDArray[np.float64],
) -> tuple[float, NDArray[np.float64], NDArray[np.float64]]:
    """
    Cross-entropy loss of the softmax classifier and its gradient.

    Args:
        inputs: Samples of shape (N, D).
        labels: Classes of shape (N,).
        weights: Weights of shape (D, K).
        bias: Bias of shape (K,).
    Returns:
        Loss, gradient of the loss w.r.t. the weights of shape (D, K) and w.r.t. the bias of shape (K,).
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
test_digit_recognition.TestLoss.eval(loss_fn=softmax_loss)

## 5. Trénování

- Natrénujte klasifikátor **metodou největšího spádu**:
  - začněte z malých náhodných vah (např. normální rozdělení se směrodatnou odchylkou 0.01) a nulového biasu,
  - v každé iteraci spočítejte gradient lossu na všech trénovacích datech (`softmax_loss`) a upravte parametry proti jeho směru
    $$
    \boldsymbol{w} := \boldsymbol{w} - \eta \cdot \frac{\partial L}{\partial \boldsymbol{w}}
    $$
    $$
    \boldsymbol{b} := \boldsymbol{b} - \eta \cdot \frac{\partial L}{\partial \boldsymbol{b}}
    $$
    kde
    - $\eta$ je krok učení (`learning_rate`).
- Výsledné parametry uložte do proměnných `weights` a `bias`.
- V každé iteraci si zaznamenejte loss a přesnost na trénovacích i validačních datech a vykreslete jejich průběh.
- Validační data slouží k volbě hyperparametrů: vyzkoušejte několik hodnot kroku učení a počtu iterací a ponechte ty, které dávají nejvyšší přesnost na validačních datech.
- Test vyžaduje, aby parametry `weights`, `bias` dosáhly přesnosti alespoň **0.95** na trénovacích a **0.93** na validačních datech.

Vzorový výstup:

<figure class="image">
  <img src="../figures/digit_recognition-expected_outputs_training.png" alt="Vzorový výstup" style="width: 9.6in"/>
</figure>

In [ ]:
########################################
# TODO: implement

# learning_rate = ...
# num_iterations = ...

# weights = ...
# bias = ...

# Vykresleni
# ...

########################################

In [ ]:
test_digit_recognition.TestAccuracy.eval(
    weights=weights,
    bias=bias,
    inputs_train=inputs_train,
    inputs_val=inputs_val,
    labels_train=labels_train,
    labels_val=labels_val,
    predict_fn=predict,
    accuracy_fn=accuracy,
)

## 6. Vyhodnocení

- Vypočtěte přesnost (accuracy) klasifikátoru na trénovacích a validačních datech a matici záměn (confusion matrix) na validačních datech.
- Vykreslete váhy jednotlivých tříd (sloupce `weights`) jako obrázky 8×8. Co představují? Vykreslete i několik chybně klasifikovaných validačních vzorků.
- Které číslice se zaměňují? Proč je validační přesnost jen odhadem přesnosti na nových datech, když se podle ní volily hyperparametry?

Vzorový výstup:

<figure class="image">
  <img src="../figures/digit_recognition-expected_outputs_evaluation.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
########################################
# TODO: implement

# train_accuracy = ...
# val_accuracy = ...
# confusion = ...

# Vykresleni
# ...

########################################

## 7. Bonus: vlastní obrázek

- Vyzkoušejte natrénovaný klasifikátor na fotografii ručně psaných číslic, např. `data/digits.jpg` (v řádku $i$ shora je číslice $i$) nebo na vlastní fotografii (tu odevzdejte spolu s notebookem).
- Implementujte funkci `extract_digits(rgb)`, která najde na fotografii číslice a vrátí dvojici `(boxes, inputs)`:
  - `boxes`: pole tvaru `(N, 4)` s obalovými obdélníky číslic `(x1, y1, x2, y2)` v pixelech,
  - `inputs`: pole tvaru `(N, 64)` se vzorky ve **stejném formátu jako `inputs_train`** ve stejném pořadí jako `boxes`.
- Doporučený postup:
  - prahování šedotónového obrázku (např. Otsu) a souvislé oblasti (`cv.connectedComponentsWithStats`), odfiltrování oblastí, které nejsou číslicemi (meze relativně k velikosti obrázku),
  - výřez číslice zmenšený se zachováním poměru stran tak, aby se vešel do obrázku 32×32, a vycentrovaný,
  - převod na obrázek 8×8 jako v datasetu: počet pixelů číslice v blocích 4×4 a stejné přeškálování jako u `inputs_train`.
- Vykreslete fotografii s obdélníky a predikovanými třídami a vzorky `inputs` jako obrázky 8×8. Proč je přesnost nižší než na validačních datech?
- Test použije `extract_digits` na `data/digits.jpg` a na jeho zmenšenou verzi a s vašimi `weights`, `bias` vyžaduje přesnost alespoň **0.50**.

Vzorový výstup:

<figure class="image">
  <img src="../figures/digit_recognition-expected_outputs_my_image.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
def extract_digits(rgb: NDArray[np.uint8]) -> tuple[NDArray[np.int64], NDArray[np.float64]]:
    """
    Find the handwritten digits in the image and convert them to the format of the digits dataset.

    Args:
        rgb: Image in RGB order of dtype uint8.
    Returns:
        Bounding boxes (x1, y1, x2, y2) of shape (N, 4) and samples of shape (N, 64) in the same format as inputs_train.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
########################################
# TODO: implement

# my_rgb = ...
# my_boxes, my_inputs = ...
# my_predicted = ...
# my_labels = ...
# my_accuracy = ...

# Vykresleni
# ...

########################################

In [ ]:
test_digit_recognition.TestOwnImage.eval(
    extract_fn=extract_digits, weights=weights, bias=bias, predict_fn=predict, accuracy_fn=accuracy
)